---
## Setup: Start vLLM + Onboard OpenClaw

Run the cell below **once**. It will:
1. Start vLLM serving Qwen3-4B on your AMD GPU (background)
2. Tail the startup logs live until the model is loaded
3. Onboard OpenClaw against the local vLLM endpoint
4. Activate the credential and set lean mode

When it finishes (green), open a **Terminal** and run `openclaw tui` to start chatting.

In [ ]:
import subprocess, time, os, re, sys
from IPython.display import clear_output
try:
    import httpx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

# ── 1. Start vLLM in background ─────────────────────────────────────
print("▶ Starting vLLM server...")
env = {**os.environ, "VLLM_USE_TRITON_FLASH_ATTN": "0"}
vllm_proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-4B",
     "--served-model-name", "Qwen3-4B",
     "--api-key", "abc-123",
     "--port", "8000",
     "--enable-auto-tool-choice",
     "--tool-call-parser", "hermes",
     "--trust-remote-code",
     "--max_model_len", "24272",
     "--gpu-memory-utilization", "0.85"],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
)

# ── 2. Tail vLLM log (latest 10 lines) + poll /health until ready ───
ready = False
for tick in range(300):  # up to ~5 min
    time.sleep(1)
    try:
        with open("/tmp/vllm.log") as f:
            lines = f.readlines()
        clear_output(wait=True)
        print("▶ Starting vLLM server... (waiting for model load)\n")
        for ln in lines[-10:]:
            print(ln.rstrip())
    except FileNotFoundError:
        pass
    # Check health every 5s
    if tick % 5 == 4:
        try:
            r = httpx.get("http://localhost:8000/health", timeout=2)
            if r.status_code == 200:
                ready = True
                break
        except Exception:
            pass

clear_output(wait=True)
if not ready:
    print("✗ vLLM did not become healthy in 5 minutes. Check /tmp/vllm.log")
    raise SystemExit(1)

print("✓ vLLM is healthy!\n")

# ── 3. Onboard OpenClaw ──────────────────────────────────────────────
print("▶ Onboarding OpenClaw...")
r = subprocess.run([
    "openclaw", "onboard", "--non-interactive", "--mode", "local",
    "--auth-choice", "custom-api-key",
    "--custom-base-url", "http://localhost:8000/v1",
    "--custom-model-id", "Qwen3-4B", "--custom-provider-id", "vllm",
    "--custom-compatibility", "openai", "--custom-api-key", "abc-123",
    "--secret-input-mode", "plaintext", "--custom-image-input",
    "--skip-channels", "--skip-search", "--skip-hooks", "--skip-ui",
    "--skip-health", "--accept-risk",
], capture_output=True, text=True)
print(r.stdout.strip())
if r.stderr.strip():
    print(r.stderr.strip())

# Activate the latest credential
match = re.search(r"(vllm:setup-[\w-]+)", r.stdout + r.stderr)
if match:
    cred_id = match.group(1)
    print(f"\n▶ Activating credential {cred_id}...")
    a = subprocess.run(
        ["openclaw", "models", "auth", "activate", cred_id, "--agent", "main"],
        capture_output=True, text=True,
    )
    print(a.stdout.strip())

# ── 4. Lean mode config ─────────────────────────────────────────────
print("\n▶ Setting lean mode...")
subprocess.run(
    ["openclaw", "config", "set",
     "agents.defaults.experimental.localModelLean", "true"],
    capture_output=True, text=True,
)
print("  Config applied.")

print("\n" + "=" * 60)
print("✓ All done! Open a Terminal and run: openclaw tui")
print("=" * 60)

---
## Chat with OpenClaw

1. Open a **Terminal** in JupyterLab: **File > New > Terminal**
2. Run: `openclaw tui`
3. Type **Hi** and press Enter to say hello!

Once it responds, try the clone & analyze prompt in the next section.

---
## Try it: Clone a Repo and Ask for Analysis

Copy and paste the prompt below into the OpenClaw chat above:

```
Clone https://github.com/matrived/blr_workshop.git in /workspace directory, show me the repository structure line-by-line one-liner description of them and also describe cell-by-cell what unsloth-finetuning/grpo_game2048-finetune-unsloth/gpt_oss_20B_Reinforcement_Learning_2048_Game_BF16.ipynb file is doing in each cell in 1 line and lastly what could be improved
```